# ProductIQ Phase 2 — Step 2.7: Canonical Schema

This notebook builds the ProductIQ canonical product dataframe from the prepared AJIO pipeline output.

## Phase objective

Define a stable ProductIQ-owned product schema with explicit field names, dtypes, and deterministic source mapping.

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from data.catalog import (
    AJIO_SOURCE_NAME,
    CANONICAL_COLUMN_ORDER,
    CANONICAL_FIELD_DEFINITIONS,
    AjioCanonicalProductBuilder,
)
from data.deduplication import AjioDataDeduplicator
from data.ingestion import AjioDataLoader
from data.preprocessing import AjioDataCleaner, AjioDataNormalizer
from data.validation import AjioDataValidator


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    msg = "Could not locate ProductIQ project root"
    raise RuntimeError(msg)


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATASET_PATH = PROJECT_ROOT / "resources" / "raw" / "Ajio Fashion Clothing.csv"

raw_df = AjioDataLoader().load(DATASET_PATH)
AjioDataValidator().validate(raw_df)
cleaned_df = AjioDataCleaner().clean(raw_df).dataframe
normalized_df = AjioDataNormalizer().normalize(cleaned_df).dataframe
prepared_df = AjioDataDeduplicator().deduplicate(normalized_df).dataframe

print("Prepared rows:", len(prepared_df))

In [ ]:
pd.DataFrame(
    [
        {
            "canonical_name": field.canonical_name,
            "source_column": field.source_column,
            "dtype": field.dtype,
        }
        for field in CANONICAL_FIELD_DEFINITIONS
    ]
)

In [ ]:
build_result = AjioCanonicalProductBuilder().build(prepared_df)
canonical_df = build_result.dataframe
print(build_result.report.summary())

In [ ]:
print("Canonical columns:", list(canonical_df.columns))
print("Expected order:", list(CANONICAL_COLUMN_ORDER))
print(canonical_df.dtypes)

In [ ]:
print("Row count preserved:", len(canonical_df) == len(prepared_df))
print("Unique product_id:", canonical_df["product_id"].nunique())
print("Duplicate product_id rows:", int(canonical_df["product_id"].duplicated().sum()))
print("Source values:", canonical_df["source"].unique())
print("Price anomalies:", int(canonical_df["price_anomaly"].sum()))

In [ ]:
sample = canonical_df.head(3)
sample[["product_id", "brand", "color_raw", "color_normalized", "discount_price_inr", "source"]]

## Phase 2.7 summary

The canonical builder maps prepared AJIO rows to 14 ProductIQ-owned fields without changing product values, row order, or row count.